In [0]:
import datetime

# Cria o campo que o Job preencherá. O padrão é a data de hoje.
dbutils.widgets.text("data_processamento", datetime.datetime.now().strftime('%Y-%m-%d'))

# Captura o valor enviado pelo Job ou preenchido manualmente
data_alvo = dbutils.widgets.get("data_processamento")

# Define os caminhos baseados no parâmetro
path_today = f"{volume_path}{data_alvo}/"

In [0]:
from pyspark.sql import functions as F
import datetime

# Configurações base
catalog = "sptrans_catalog"
schema = "gtfs_data"
volume_path = f"dbfs:/Volumes/{catalog}/{schema}/landing_zone/"

# configurações de checkpoint
checkpoint_base = f"dbfs:/Volumes/{catalog}/{schema}/_checkpoints/bronze/"

# pega a data atual 
today = datetime.datetime.now().strftime('%Y-%m-%d')
path_today = f"{volume_path}{today}/"

# Lista de arquivos que serão processados
gtfs_files = [
    "agency",
    "calendar",
    "fare_attributes",
    "fare_rules",
    "frequencies",
    "routes",
    "shapes",
    "stop_times",
    "stops",
    "trips",
]

# Loop de ingestão
for file in gtfs_files:
    print(f"Iniciando a ingestão do arquivo: {file}")
    
    target_table = f"{catalog}.{schema}.bronze_{file}"
    checkpoint_path = f"{checkpoint_base}{file}"

    # Leitura com Auto Loader
    df_bronze = (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")  
        .option("cloudFiles.inferColumnTypes", "true")
        .option("pathGlobFilter", f"{file}.txt")
        .option("recursiveFileLookup", "true")
        .option("cloudFiles.schemaLocation", checkpoint_path)
        .load(volume_path))
  
    # Escrita no Unity Catalog
    (df_bronze.writeStream
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(target_table))

    print(f"Ingestão do arquivo {file} concluída com sucesso!")


In [0]:
# Converte o relatório para Spark DataFrame e salva no Unity Catalog
df_audit = spark.createDataFrame(relatorio)
df_audit = df_audit.withColumn("data_execucao", F.current_timestamp())

# Salva como log histórico
df_audit.write.format("delta").mode("append").saveAsTable(f"{catalog}.{schema}.audit_log_ingestion")